# 03 — Modelo 1: Reglas de Asociación (FP-Growth)
## Cross-Selling Recommender | Proyecto Final Henry

**Por qué FP-Growth sobre Apriori:**
FP-Growth construye un árbol comprimido (FP-tree) → O(n) vs O(2^k) de Apriori.
Con 200K órdenes y ~15K productos únicos, FP-Growth es 10–50× más rápido.

**Umbrales:**
| Umbral | Valor | Interpretación |
|---|---|---|
| min_support | 0.01 | Par en ≥ 2 000 cestas → robusto |
| min_confidence | 0.30 | 30% de veces A implica B → útil |
| min_lift | 1.2 | 20% más frecuente que azar |


In [ ]:
# ── 1. Configuración ────────────────────────────────────────────────────────
import sys, logging
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_config, load_instacart_tables, build_master_dataframe
from src.data.preprocessor import run_cleaning_pipeline
from src.models.association_rules import (
    build_transactions_list, encode_transactions, train_fpgrowth,
    get_recommendations_for_product, save_rules, summarize_rules,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


---
## 2. Carga y Preprocesamiento

In [ ]:
# ── 2.1 Cargar tablas ────────────────────────────────────────────────────────
cfg = load_config()
orders, op_prior, products, aisles, departments = load_instacart_tables(cfg)

df_master = build_master_dataframe(
    orders=orders, order_products=op_prior,
    products=products, aisles=aisles,
    departments=departments, eval_set='prior',
)
product_names = products.set_index('product_id')['product_name'].to_dict()
print(f'DataFrame maestro: {df_master.shape}')
print(f'Productos en catálogo: {len(product_names):,}')


In [ ]:
# ── 2.2 Pipeline — muestra 200K órdenes para FP-Growth ───────────────────────
# run_cleaning_pipeline retorna (df_full, df_apriori)
# df_apriori = 200K órdenes muestreadas aleatoriamente
_, df_apriori = run_cleaning_pipeline(
    df=df_master,
    min_basket_items=2,
    add_temporal=True,
    apriori_sample=200_000,
    random_state=RANDOM_STATE,
)
print(f'df_apriori: {df_apriori.shape}')
print(f'Órdenes únicas: {df_apriori["order_id"].nunique():,}')
print(f'Productos únicos: {df_apriori["product_id"].nunique():,}')


---
## 3. Preparación de Transacciones

In [ ]:
# ── 3.1 Lista de transacciones ───────────────────────────────────────────────
transactions = build_transactions_list(df_apriori)
basket_sizes = [len(t) for t in transactions]
print(f'Transacciones   : {len(transactions):,}')
print(f'Media basket    : {np.mean(basket_sizes):.2f} ítems')
print(f'Ítems únicos    : {len(set(p for t in transactions for p in t)):,}')


In [ ]:
# ── 3.2 Encoding binario (TransactionEncoder) ────────────────────────────────
basket_matrix = encode_transactions(transactions)
print(f'Matriz: {basket_matrix.shape[0]:,} × {basket_matrix.shape[1]:,}')
print(f'Sparsity: {1 - basket_matrix.values.mean():.4%}')
print(f'Memoria: {basket_matrix.memory_usage(deep=True).sum() / 1e6:.1f} MB')


---
## 4. Entrenamiento FP-Growth

In [ ]:
# ── 4.1 Entrenar FP-Growth + generar reglas ──────────────────────────────────
rules = train_fpgrowth(
    basket_matrix=basket_matrix,
    min_support=0.01,
    min_confidence=0.30,
    min_lift=1.2,
    max_len=3,
)
summarize_rules(rules)
rules[['antecedents','consequents','support','confidence','lift']].head(10)


---
## 5. Exploración de Reglas

In [ ]:
# ── 5.1 Distribución de métricas ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribución de métricas — Reglas de Asociación', fontsize=13, fontweight='bold')
for ax, (col, color) in zip(axes, [('support','#4C72B0'),('confidence','#DD8452'),('lift','#55A868')]):
    ax.hist(rules[col], bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(rules[col].median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mediana: {rules[col].median():.3f}')
    ax.set_xlabel(col.capitalize()); ax.set_ylabel('Frecuencia')
    ax.set_title(f'Distribución de {col.capitalize()}'); ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '09_ar_metrics_distribution.png', bbox_inches='tight')
plt.show()
print('Guardado: outputs/figures/09_ar_metrics_distribution.png')


In [ ]:
# ── 5.2 Scatter: Support vs Confidence coloreado por Lift ────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(rules['support'], rules['confidence'], c=rules['lift'],
                cmap='YlOrRd', alpha=0.6, s=15, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Lift')
ax.set_xlabel('Support'); ax.set_ylabel('Confidence')
ax.set_title('Support vs Confidence coloreado por Lift', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '10_ar_support_confidence_lift.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.3 Top 20 reglas por Lift ────────────────────────────────────────────────
def fmt_rule(row):
    a = ' + '.join([product_names.get(int(x), str(x)) for x in row['antecedents']])
    c = ' + '.join([product_names.get(int(x), str(x)) for x in row['consequents']])
    return f'{a}  →  {c}'

top20 = rules.head(20).copy()
top20['label'] = top20.apply(fmt_rule, axis=1)
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(top20['label'][::-1], top20['lift'][::-1],
               color=sns.color_palette('YlOrRd', n_colors=20), edgecolor='white')
ax.set_xlabel('Lift'); ax.set_title('Top 20 Reglas por Lift', fontsize=13, fontweight='bold')
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, label='Lift=1 (aleatorio)')
ax.legend()
for bar, val in zip(bars, top20['lift'][::-1]):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left', fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '11_ar_top20_lift.png', bbox_inches='tight')
plt.show()


---
## 6. Recomendaciones de Cross-Selling

In [ ]:
# ── 6.1 Top antecedentes con más cobertura de reglas ─────────────────────────
all_ant = [int(x) for s in rules['antecedents'] for x in s]
top_ant = Counter(all_ant).most_common(10)
print('Top 10 productos como antecedentes:')
for pid, cnt in top_ant:
    print(f'  {pid:>8} | {product_names.get(pid, str(pid)):<45} | {cnt} reglas')


In [ ]:
# ── 6.2 Demo de recomendaciones para el producto con más cobertura ────────────
best_pid  = top_ant[0][0]
best_name = product_names.get(best_pid, f'ID {best_pid}')
recs = get_recommendations_for_product(
    product_id=best_pid, rules=rules,
    product_names=product_names, top_k=10,
)
print(f'Top-10 recomendaciones para: {best_name} (ID={best_pid})')
print(recs.to_string(index=False))


---
## 7. Cobertura de Catálogo

In [ ]:
# ── 7.1 Cobertura: qué % del catálogo aparece en alguna regla ────────────────
prods_in_rules = set()
for s in rules['antecedents']: prods_in_rules.update([int(x) for x in s])
for s in rules['consequents']: prods_in_rules.update([int(x) for x in s])
total_prods = products['product_id'].nunique()
cov = len(prods_in_rules) / total_prods * 100
print(f'Catálogo total      : {total_prods:,}')
print(f'Productos en reglas : {len(prods_in_rules):,}')
print(f'Cobertura           : {cov:.1f}%')
print()
print('Baja cobertura es esperada con min_support=0.01 (la long tail queda fuera).')
print('El modelo ALS del siguiente notebook complementa este gap.')


---
## 8. Guardado del Modelo

In [ ]:
# ── 8.1 Guardar reglas ───────────────────────────────────────────────────────
path = save_rules(rules, filename='fpgrowth_rules.pkl')
print(f'Reglas guardadas: {path}')
print(f'Total reglas    : {len(rules):,}')
print(f'Tamaño archivo  : {path.stat().st_size / 1024:.1f} KB')
print('\n-> Siguiente: notebook 04_collaborative_filter.ipynb')
